# Inter-Annotator Agreement (IAA) Analysis

Measures agreement between the three annotators (**MC**, **ME**, **TC**) who screened a
sample of papers for inclusion, and merges their verdicts into a single table.

The screening subsets were produced in [02_data_merge.ipynb](02_data_merge.ipynb). The IAA
design is:

- **20 papers** reviewed by **all 3** annotators (`iaa_set == "all"`)
- **10 papers per pair** reviewed by exactly 2 annotators (`iaa_set == "pair_XX_YY"`)

So every **pair shares 30 papers** (20 + 10), all three share **20**, and there are **50
unique papers** total (20 by all 3, 30 by exactly 2).

The variable of interest is **`Include`** (boolean): whether the annotator would include
the paper. The completed annotations live in `../data/processed/iaa/annotated/`.


In [1]:
import sys
sys.path.append("../")  # go up one level from notebooks/ to project root

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import krippendorff
from sklearn.metrics import cohen_kappa_score

ANNOTATORS = ["MC", "ME", "TC"]
# Per-annotator columns (vary by annotator); "Include" is the variable of interest.
ANNOTATION_COLS = ["Include", "To discuss", "Research area", "Topic/Ideas", "Notes"]
# Paper-level columns (identical across annotators for a given paper).
META_COLS = ["iaa_set", "ids", "year", "sources", "title", "abstract",
             "published_in", "paper_type"]

iaa_dir = Path("../data/processed/iaa/annotated")

## Load annotated files

In [2]:
dfs = {a: pd.read_excel(iaa_dir / f"annotator_{a}_annotated.xlsx") for a in ANNOTATORS}

for a, df in dfs.items():
    assert len(df) == 40, f"{a} has {len(df)} rows (expected 40)"
    assert df["Include"].notna().all(), f"{a} has missing Include values"
    print(f"Annotator {a}: {len(df)} rows  "
          f"iaa_set={df['iaa_set'].value_counts().to_dict()}")

Annotator MC: 40 rows  iaa_set={'all': 20, 'pair_MC_ME': 10, 'pair_MC_TC': 10}
Annotator ME: 40 rows  iaa_set={'all': 20, 'pair_MC_ME': 10, 'pair_ME_TC': 10}
Annotator TC: 40 rows  iaa_set={'all': 20, 'pair_ME_TC': 10, 'pair_MC_TC': 10}


## Wide `Include` matrix

One row per paper, one column per annotator. `NaN` means the annotator did not review that
paper. This backs the agreement metrics and the derived counts.

In [3]:
long = pd.concat(
    [df[["paper_id", "Include"]].assign(annotator=a) for a, df in dfs.items()],
    ignore_index=True,
)
include_wide = long.pivot(index="paper_id", columns="annotator", values="Include")[ANNOTATORS]
print(f"Unique papers reviewed: {len(include_wide)}")
include_wide.head()

Unique papers reviewed: 50


annotator,MC,ME,TC
paper_id,,,
031fcedf9970f53c,False,False,NaN
06e5f45d08bfd760,NaN,True,False
0eea50f3d58ec619,False,False,False
1aecf98d861b4aea,NaN,True,False
225076b0055d9170,False,False,False


## 1. Cohen's kappa per pair of annotators

Each pair shares **30** papers (20 reviewed by all + 10 pairwise). 

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed agreement and $p_e$ the agreement expected by chance from the annotators' marginal label frequencies.

Our results show fair agreement in general.

In [4]:
kappa_rows = []
for x, y in combinations(ANNOTATORS, 2):
    both = include_wide[[x, y]].dropna()  # the 30 papers reviewed by both
    k = cohen_kappa_score(both[x].astype(int), both[y].astype(int))
    kappa_rows.append({"pair": f"{x}-{y}", "n_shared": len(both), "cohen_kappa": k})

kappa_df = pd.DataFrame(kappa_rows).round(3)
print(kappa_df[["pair", "cohen_kappa"]])
kappa_df


    pair  cohen_kappa
0  MC-ME        0.384
1  MC-TC        0.348
2  ME-TC        0.342


,pair,n_shared,cohen_kappa
0,MC-ME,30,0.384
1,MC-TC,30,0.348
2,ME-TC,30,0.342


## 2. Krippendorff's alpha (3 annotators)

Computed on the **20 papers** reviewed by all three annotators. The reliability matrix has shape `3 raters × 20 units`.

In [5]:
all_papers = include_wide.dropna()  # 20 papers reviewed by all three
reliability = all_papers[ANNOTATORS].astype(int).T.values  # 3 x 20

alpha = krippendorff.alpha(reliability_data=reliability, level_of_measurement="nominal")
print(f"Papers reviewed by all three: {len(all_papers)}")
print(f"Krippendorff's alpha (nominal): {alpha:.3f}")

Papers reviewed by all three: 20
Krippendorff's alpha (nominal): 0.454


## 3. Merged annotations table

One row per paper (50 total), carrying the paper metadata once plus every annotator's
`Include`, `To discuss`, `Research area`, `Topic/Ideas` and `Notes` (suffixed by annotator
code). `NaN` in an annotator's columns means they did not review that paper.

Derived columns:
- `n_reviewers` — how many annotators reviewed the paper,
- `n_include` — how many marked it for inclusion,
- `disagreement` — `True` when the reviewers are **not unanimous** (at least two disagree),
- `any_to_discuss` — `True` when at least one reviewer flagged the paper "To discuss".

In [ ]:
# Paper-level columns once (identical across annotators for a given paper).
meta = (pd.concat([df[["paper_id"] + META_COLS] for df in dfs.values()], ignore_index=True)
        .drop_duplicates(subset="paper_id"))

# Per-annotator annotation columns, suffixed by annotator code.
ann_frames = [
    df[["paper_id"] + ANNOTATION_COLS]
    .rename(columns={c: f"{a}_{c}" for c in ANNOTATION_COLS})
    .set_index("paper_id")
    for a, df in dfs.items()
]
ann_wide = ann_frames[0].join(ann_frames[1:], how="outer")

merged = meta.set_index("paper_id").join(ann_wide, how="outer").reset_index()

# Derived columns from the per-annotator Include verdicts.
inc_cols = [f"{a}_Include" for a in ANNOTATORS]
merged["n_reviewers"] = merged[inc_cols].notna().sum(axis=1)
merged["n_include"] = (merged[inc_cols] == True).sum(axis=1)
merged["disagreement"] = (merged["n_include"] > 0) & (merged["n_include"] < merged["n_reviewers"])

# True if at least one reviewer flagged the paper "To discuss".
disc_cols = [f"{a}_To discuss" for a in ANNOTATORS]
merged["any_to_discuss"] = (merged[disc_cols] == True).any(axis=1)

# Column order: identifiers, per-annotator Include, derived flags, rest of metadata, notes.
note_cols = [f"{a}_{c}" for c in ANNOTATION_COLS if c != "Include" for a in ANNOTATORS]
ordered = (["paper_id", "title", "year"] + inc_cols
           + ["n_reviewers", "n_include", "disagreement", "any_to_discuss"]
           + [c for c in META_COLS if c not in ("title", "year")] + note_cols)
merged = merged[ordered]

print(f"Merged table: {merged.shape[0]} papers x {merged.shape[1]} columns")
merged[["paper_id", "title"] + inc_cols
       + ["n_reviewers", "n_include", "disagreement", "any_to_discuss"]]

In [7]:
output_path = iaa_dir / "iaa_merged.xlsx"
merged.to_excel(output_path, index=False)
print(f"Wrote {output_path}")

Wrote ../data/processed/iaa/annotated/iaa_merged.xlsx


## 4. Basic statistics

In [10]:
n_total = len(merged)
n_any_include = int((merged["n_include"] >= 1).sum())
n_unanimous_include = int((merged["n_include"] == merged["n_reviewers"]).sum())
n_disagreement = int(merged["disagreement"].sum())
n_unanimous_exclude = int((merged["n_include"] == 0).sum())

print(f"Total unique papers reviewed                        : {n_total}")
print(f"Papers with at least one include                    : {n_any_include}")
print(f"Papers included by all reviewers who checked it     : {n_unanimous_include}")
print(f"Papers with disagreement                            : {n_disagreement}")
print(f"Papers unanimously excluded                         : {n_unanimous_exclude}")

Total unique papers reviewed                        : 50
Papers with at least one include                    : 30
Papers included by all reviewers who checked it     : 10
Papers with disagreement                            : 20
Papers unanimously excluded                         : 20
